In [ ]:
# ==========================================
# Notebook 5: Feature Engineering & Preprocessing
# ==========================================
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Chargement des trois splits
train_df = pd.read_parquet("artifacts/03_train.parquet")
val_df = pd.read_parquet("artifacts/03_val.parquet")
test_df = pd.read_parquet("artifacts/03_test.parquet")

print(f"Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")

Train: (67534, 23), Val: (14472, 23), Test: (14472, 23)


In [2]:
# 1. Création des variables (visibles uniquement au moment de la commande)
def create_features(df):
    df = df.copy()
    
    # Dates en datetime
    df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
    
    # Features temporelles
    df['purchase_year'] = df['order_purchase_timestamp'].dt.year
    df['purchase_month'] = df['order_purchase_timestamp'].dt.month
    df['purchase_day'] = df['order_purchase_timestamp'].dt.day
    df['purchase_dayofweek'] = df['order_purchase_timestamp'].dt.dayofweek
    df['purchase_hour'] = df['order_purchase_timestamp'].dt.hour
    df['is_weekend'] = df['purchase_dayofweek'].isin([5, 6]).astype(int)
    
    # Features géographiques
    if 'customer_state' in df.columns and 'seller_state' in df.columns:
        df['same_state'] = (df['customer_state'] == df['seller_state']).astype(int)
    else:
        df['same_state'] = 0
        
    # Ratios prix et fret par article
    if 'total_items' in df.columns:
        df['price_per_item'] = df['total_price'] / np.maximum(df['total_items'], 1)
        df['freight_per_item'] = df['total_freight'] / np.maximum(df['total_items'], 1)
        
    return df

# Application de la création de features sur tous les splits
train_feat = create_features(train_df)
val_feat = create_features(val_df)
test_feat = create_features(test_df)

print("Features créées avec succès !")

Features créées avec succès !


In [3]:
# 1. Définition des colonnes
target_col = 'is_late'

num_cols = [
    'total_items', 'total_price', 'total_freight', 
    'avg_product_weight_g', 'price_per_item', 'freight_per_item',
    'purchase_month', 'purchase_day', 'purchase_dayofweek', 'purchase_hour'
]

cat_cols = ['customer_state', 'seller_state', 'main_payment_type']
bool_cols = ['is_weekend', 'same_state']

feature_list = num_cols + cat_cols + bool_cols

# Séparation des caractéristiques (X) et de la cible (y)
X_train, y_train = train_feat[feature_list], train_feat[target_col]
X_val, y_val = val_feat[feature_list], val_feat[target_col]
X_test, y_test = test_feat[feature_list], test_feat[target_col]

# Pipelines de pré-traitement
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols),
    ('passthrough', 'passthrough', bool_cols)
])

# FIT uniquement sur le jeu de TRAIN pour éviter toute fuite de données
print("Ajustement du pré-processeur...")
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

# Récupération des noms de colonnes
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
cat_encoded_names = list(cat_encoder.get_feature_names_out(cat_cols))
all_feature_names = num_cols + cat_encoded_names + bool_cols

# Reconstruction des DataFrames
X_train_df = pd.DataFrame(X_train_processed, columns=all_feature_names)
X_val_df = pd.DataFrame(X_val_processed, columns=all_feature_names)
X_test_df = pd.DataFrame(X_test_processed, columns=all_feature_names)

X_train_df[target_col] = y_train.values
X_val_df[target_col] = y_val.values
X_test_df[target_col] = y_test.values

print("Transformation terminée !")

Ajustement du pré-processeur...
Transformation terminée !


In [4]:
# 4. Sauvegarde des modèles et données transformées
os.makedirs("artifacts/models", exist_ok=True)

# Objets sklearn ajustés (fit)
joblib.dump(preprocessor, "artifacts/models/fitted_preprocessor.joblib")
joblib.dump(all_feature_names, "artifacts/models/feature_names.joblib")

# Tables finales
X_train_df.to_parquet("artifacts/05_train_features.parquet", index=False)
X_val_df.to_parquet("artifacts/05_val_features.parquet", index=False)
X_test_df.to_parquet("artifacts/05_test_features.parquet", index=False)

print("✅ Artefacts enregistrés avec succès dans le dossier 'artifacts/' !")

✅ Artefacts enregistrés avec succès dans le dossier 'artifacts/' !
